# 01 - Data Cleaning and Merging
The purpose of this notebook is to load, clean, and, merge the raw workout and bodyweight datasets. The primary steps include standardizing data types, handling missing values, and combining the two datasets on the "Date" column. The final output is a clean, master DataFrame that will be used for subsequent in-depth analysis and visualization.

In [81]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)        # Widen the display

In [82]:
# loading csv files
workouts_df = pd.read_csv('../data/workout_data.csv')
weight_df = pd.read_csv('../data/measurement_data.csv')

### 2.1 - Previewing the Workout Data
To get a better idea of the way this data is organized, let's view the head along with any attributes with null values. This will help me decide what data I can safely delete.

In [83]:
# view first few rows
workouts_df.head()

,title,start_time,end_time,description,exercise_title,superset_id,exercise_notes,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
0,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Single Leg Press (Machine),0.0,egym lowkey feel lighter,0,normal,50.0,8.0,NaN,NaN,8.5
1,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Single Leg Press (Machine),0.0,egym lowkey feel lighter,1,normal,80.0,4.0,NaN,NaN,10.0
2,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Leg Press Horizontal (Machine),0.0,egym,0,normal,200.0,8.0,NaN,NaN,8.0
3,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Leg Press Horizontal (Machine),0.0,egym,1,normal,250.0,8.0,NaN,NaN,9.0
4,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Seated Leg Curl (Machine),NaN,NaN,0,warmup,50.0,8.0,NaN,NaN,6.0


In [84]:
# checking data types
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2104 entries, 0 to 2103
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2104 non-null   object 
 1   start_time        2104 non-null   object 
 2   end_time          2104 non-null   object 
 3   description       314 non-null    object 
 4   exercise_title    2104 non-null   object 
 5   superset_id       78 non-null     float64
 6   exercise_notes    130 non-null    object 
 7   set_index         2104 non-null   int64  
 8   set_type          2104 non-null   object 
 9   weight_lbs        1957 non-null   float64
 10  reps              2042 non-null   float64
 11  distance_miles    11 non-null     float64
 12  duration_seconds  62 non-null     float64
 13  rpe               2004 non-null   float64
dtypes: float64(6), int64(1), object(7)
memory usage: 230.3+ KB


In [85]:
workouts_df.isnull().sum()

title                  0
start_time             0
end_time               0
description         1790
exercise_title         0
superset_id         2026
exercise_notes      1974
set_index              0
set_type               0
weight_lbs           147
reps                  62
distance_miles      2093
duration_seconds    2042
rpe                  100
dtype: int64

### 2.2 - Removing Metadata
We can immediately see some metadata that will not be necessary in our data analysis, so we can remove them. This includes the description and exercise notes. The title of the workout itself is also not strictly necessary in the data analysis, but may make it easier to organize specific exercises done on each day into their respective muscle groups.

In [86]:
workouts_df = workouts_df.drop(['description', 'exercise_notes'], axis=1)

### 2.3 - Verifying Irrelevant Exercise Data
Since we aim to look at strength training data, we can get rid of any data related to cardio or other exercise types. These exercises will not have weight or rep information. However, some exercises, like pull ups, may not have weight information since it is based on bodyweight. These exercises are vital for checking trends in strength so they should not be deleted. Thus, we will analyze the data closer to verify what data should and should not be deleted.

In [87]:
workouts_df[workouts_df['weight_lbs'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.0,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.0,NaN
19,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Chin Up,NaN,0,normal,NaN,12.0,NaN,NaN,8.5
20,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Chin Up,NaN,1,normal,NaN,8.0,NaN,NaN,9.5
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2037,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Hanging Leg Raise,NaN,0,normal,NaN,4.0,NaN,NaN,10.0
2038,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Hanging Leg Raise,NaN,1,normal,NaN,4.0,NaN,NaN,10.0
2039,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Dead Hang,NaN,0,normal,NaN,NaN,NaN,34.0,NaN
2040,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Treadmill,NaN,0,normal,NaN,NaN,0.32,900.0,NaN


In [88]:
workouts_df[workouts_df['reps'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.0,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.0,NaN
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.0,NaN
136,Arms,"2 Aug 2025, 15:38","2 Aug 2025, 17:08",Spinning,NaN,0,normal,NaN,NaN,NaN,300.0,NaN
165,Arms,"30 Jul 2025, 15:06","30 Jul 2025, 16:13",Spinning,NaN,0,normal,NaN,NaN,NaN,300.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1899,Core/Cardio,"30 Oct 2024, 12:56","30 Oct 2024, 14:01",Plank,NaN,1,normal,NaN,NaN,NaN,44.0,NaN
1900,Core/Cardio,"30 Oct 2024, 12:56","30 Oct 2024, 14:01",Treadmill,NaN,0,normal,NaN,NaN,0.25,572.0,NaN
2039,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Dead Hang,NaN,0,normal,NaN,NaN,NaN,34.0,NaN
2040,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Treadmill,NaN,0,normal,NaN,NaN,0.32,900.0,NaN
